# Solution: Convolutions and Pooling

## Introduction

In this notebook, we will explore the concepts of *convolution* and *pooling* in deep learning. 

We will start by manually implementing these operations to gain a deeper understanding of how they work. 
Later, we will use PyTorch's built-in modules to see the advantages of using such libraries.

This is an interactive notebook, so download it and play with the widgets.

You also have an exercise at the end of the notebook!

## Libraries

Let's begin by importing the necessary libraries. 

We'll need NumPy for numerical operations, PyTorch for deep learning operations, and Matplotlib for visualization.
Furthermore, we'll need PIL for visualizing images and tkinter to be able to select images.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

import ipywidgets as widgets
from ipywidgets.widgets import VBox, HBox
from IPython.display import display, clear_output

from cycler import cycler
import seaborn as sns

# Set the color scheme
sns.set_theme()
colors = ['#0076C2', '#EC6842', '#A50034', '#009B77', '#FFB81C', '#E03C31', '#6CC24A', '#EF60A3', '#0C2340', '#00B8C8', '#6F1D77']
plt.rcParams['axes.prop_cycle'] = cycler(color=colors)
plt.rcParams.update({'font.size': 12})

## Let's use a sample Image

Load a sample image that will be used throughout the notebook. 
You can use any image you prefer or the one we propose.

In [ ]:
# Load the sample image and normalize it
sample_image = Image.open("../../../images/cracks.jfif")

sample_image = np.array(sample_image)/255 # Normalize range of values to [0,1]

plt.imshow(sample_image)
plt.axis('off');

## Convolution Operation

What is convolution?

Convolution is an operation in which a kernel (or filter) matrix, in blue, slides over an input image and is pointwise multiplied by the entries of the kernel with the overlapping values in the input image.
The sum of these values corresponds to the respective entry in the output matrix Y.

<img src="../../../images/convolution_kernel.png" width="800" float="center"/>

In mathematical terms, we can write the convolution of an input matrix $A \in \mathbb{R}^{S \times T}$ by a kernel matrix $K \in \mathbb{R}^{M \times N}$ as:
$$
(A * K)_{s,t} = \sum_{m=1}^{M} \sum_{n=1}^{N} A_{m,n} \cdot K_{s - m, t - n}
$$
where M and N are the height and width of the kernel.

### Manual implementation

Here we implement a function to manually perform the convolution operation, following the previous equation. 

Then, we use a simple 3x3 kernel as an example.

In [ ]:
# Manual Implementation of Convolution Operation
def manual_convolution(image, kernel):
    height, width, channels = image.shape
    kernel_size = kernel.shape[0]

    # we assume here that there is no padding
    feature_map = np.zeros((height - kernel_size + 1, width - kernel_size + 1, channels))

    for k in range(channels):
        for i in range(height - kernel_size + 1):
            for j in range(width - kernel_size + 1):
                patch = image[i:i+kernel_size, j:j+kernel_size]
                feature_map[i, j, k] = np.sum(patch * kernel)

    return feature_map.sum(-1) # summing over the channels

In [ ]:
# Define a simple 3x3 kernel
# The following kernel is called vertical Sobel operator and can be used to detect vertical edges
kernel = np.array([[-1, -2, -1],
                   [0, 0, 0],
                   [1, 2, 1]])

# Perform manual convolution
feature_map_manual = manual_convolution(sample_image, kernel)

In [ ]:
# Compare and visualize the results
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Original image")
plt.imshow(sample_image, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Manual Convolution")
plt.imshow(feature_map_manual, cmap='gray')
plt.axis('off')

plt.show()

### Pytorch implementation

Our manual implementation works, but for large images it's not efficient (because of the for loops).
We can equivalently use PyTorch's built-in convolutional layer (**nn.Conv2d**) to perform the convolution operation on the same sample image.

In [ ]:
# Convert the image to a PyTorch tensor
sample_image_tensor = torch.Tensor(sample_image).permute(2, 0, 1).unsqueeze(0)

# Create a convolutional layer
conv_layer = nn.Conv2d(in_channels=3, out_channels=1, kernel_size=3, 
                       stride=1, padding=0, bias=False)

conv_layer.weight = nn.Parameter(torch.FloatTensor(np.array([[kernel.T, kernel.T, kernel.T]])))

# Perform convolution using PyTorch
output = conv_layer(sample_image_tensor).detach()

In [ ]:
# Compare and visualize the results
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title("Original image")
plt.imshow(sample_image, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Pytorch Convolution")
plt.imshow(output[0, 0], cmap='gray')
plt.axis('off')

plt.show()

### Try out different kernels

You can either click on some predifined kernels or you can manually enter the values you want to assign to your kernel and see what they do.

In [ ]:
sobel_kernel_vertical_button = widgets.Button(description="Vertical Sobel kernel", disabled=False, layout=widgets.Layout(width='200px'))
sobel_kernel_horizontal_button = widgets.Button(description="Horizontal Sobel kernel", layout=widgets.Layout(width='200px'))
sharpen_kernel_button = widgets.Button(description="Sharpen kernel", layout=widgets.Layout(width='200px'))
box_blur_kernel_button = widgets.Button(description="Box Blur kernel", layout=widgets.Layout(width='200px'))
laplacian_kernel_button = widgets.Button(description="Laplacian kernel", layout=widgets.Layout(width='200px'))
emboss_kernel_button = widgets.Button(description="Emboss kernel", layout=widgets.Layout(width='200px'))
gaussian_blur_kernel_button = widgets.Button(description="Gaussian Blur kernel", layout=widgets.Layout(width='200px'))
edge_detection_kernel_button = widgets.Button(description="Edge detection kernel", layout=widgets.Layout(width='200px'))
manual_kernel_button = widgets.Button(description="Manual kernel (click here after changing the values)", layout=widgets.Layout(width='600px'), button_style='info')


def sobel_kernel_vertical(_):
    global kernel, kernel_name
    kernel_name = 'Sobel vertical'
    kernel = np.array([[-1, -2, -1],
                    [0, 0, 0],
                    [1, 2, 1]])
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
sobel_kernel_vertical_button.on_click(sobel_kernel_vertical)


def sobel_kernel_horizontal(_):
    global kernel, kernel_name
    kernel_name = 'Sobel horizontal'
    kernel = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]])
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
sobel_kernel_horizontal_button.on_click(sobel_kernel_horizontal)


def sharpen_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Sharpen'
    kernel = np.array([[0, -1, 0],
                    [-1, 5, -1],
                    [0, -1, 0]])
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
sharpen_kernel_button.on_click(sharpen_kernel)


def box_blur_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Box blur'
    kernel = np.array([[1, 1, 1],
                    [1, 1, 1],
                    [1, 1, 1]]) / 9  # Normalize the kernel to sum to 1
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
box_blur_kernel_button.on_click(box_blur_kernel)


def laplacian_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Laplacian'
    kernel = np.array([[0, 1, 0],
                    [1, -4, 1],
                    [0, 1, 0]])
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
laplacian_kernel_button.on_click(laplacian_kernel)


def emboss_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Emboss'
    kernel = np.array([[-2, -1, 0],
                    [-1, 1, 1],
                    [0, 1, 2]])
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
emboss_kernel_button.on_click(emboss_kernel)


def gaussian_blur_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Gaussian blur'
    kernel = np.array([[1, 2, 1],
                    [2, 4, 2],
                    [1, 2, 1]]) / 16  # Normalize the kernel to sum to 1
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
gaussian_blur_kernel_button.on_click(gaussian_blur_kernel)


def edge_detection_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Edge detection'
    kernel = np.array([[1, 1, 1],
                    [1, -8, 1],
                    [1, 1, 1]])
    for i, item in enumerate(items):
        item.value = kernel.reshape(-1)[i]
edge_detection_kernel_button.on_click(edge_detection_kernel)

def manual_kernel(_):
    global kernel, kernel_name
    kernel_name = 'Manual kernel'
    kernel = np.array([[0, 0, 0],
                    [0, 0, 0],
                    [0, 0, 0]])
    update_kernel()
manual_kernel_button.on_click(manual_kernel)

button_grid = VBox([HBox([sobel_kernel_vertical_button, sobel_kernel_horizontal_button, sharpen_kernel_button]), 
                    HBox([box_blur_kernel_button, laplacian_kernel_button, emboss_kernel_button]),
                    HBox([gaussian_blur_kernel_button, edge_detection_kernel_button]),
                    HBox([manual_kernel_button])])

items = [widgets.FloatText(i, layout=widgets.Layout(width='50px')) for i in kernel.reshape(-1)]

kernel_box = widgets.GridBox(items, layout=widgets.Layout(grid_template_columns="repeat(3, 50px)", 
                                            justify_content="flex-start"))

def update_kernel():
    global kernel
    kernel = np.array([item.value for item in items]).reshape(3, 3)

def run_convolution():
    global output
    conv_layer.weight = nn.Parameter(torch.FloatTensor(np.array([[kernel.T, kernel.T, kernel.T]])))
    output = conv_layer(sample_image_tensor).detach()

figure = widgets.Output()
def update_and_display_image(_):
    run_convolution()
    clear_output()
    with figure:
        fig, [ax0, ax1] = plt.subplots(1, 2, figsize=(12, 6))

        ax0.set_title("Original image")
        ax0.imshow(sample_image, cmap='gray')
        ax0.axis('off')

        ax1.imshow(output[0, 0], cmap='gray')
        ax1.set_title(f"Convolution ({kernel_name})")
        ax1.axis('off')
        
# Create a widget that triggers the image update
update_image_button = widgets.Button(description="Update Image")
update_image_button.on_click(update_and_display_image)

In [ ]:
display(button_grid)
display(kernel_box)
display(update_image_button)

## Pooling Operation

### Manual implementation

Implement a function to manually perform max pooling operation.
You can take inspiration from the manual implementation of the convolutional kernel from before.
Now, instead of pointwise multiplying the image patch by a kernel, you are extract the maximum (or average) value from the patch.


<img src="../../../images/maxpooling.png" width="800" float="center"/>

*Image source: "https://computersciencewiki.org/index.php/File:MaxpoolSample2.png"*


In [ ]:
# Manual Implementation of Pooling Operation (Max-Pooling)
def manual_max_pooling(image, pool_size):
    height, width, channels = image.shape
    pooled_height = height // pool_size
    pooled_width = width // pool_size
    pooled_image = np.zeros((channels, pooled_height, pooled_width))

    # ---------------------- student exercise --------------------------------- #
    for k in range(channels):
        for i in range(pooled_height):
            for j in range(pooled_width):
                patch = image[i*pool_size:i*pool_size+pool_size, j*pool_size:j*pool_size+pool_size, k]
                pooled_image[k, i, j] = np.max(patch) # You can change the opartion here to obtain average pooling as well
    # ---------------------- student exercise --------------------------------- #
    
    return pooled_image

# Perform manual max-pooling
pool_size = 2
pooled_image_manual = manual_max_pooling(sample_image, pool_size)

### Pytorch implementation

Use PyTorch's nn.MaxPool2d or nn.AvgPool2d to perform the pooling operation on the feature maps obtained in the previous sections.

In [ ]:
sample_image_tensor = torch.Tensor(sample_image).permute(2, 0, 1)

# Create a pooling layer
pool_layer = nn.MaxPool2d(kernel_size=pool_size)

# Perform pooling using PyTorch
pooled_output = pool_layer(sample_image_tensor).detach()

In [ ]:
assert np.isclose(pooled_image_manual, pooled_output.numpy()).all(), "Your implementation is not correct"

### Comparison

Compare the pooled feature maps obtained from the manual implementation and the built-in module, similar to the comparison for convolution.


In [ ]:
# Compare and visualize the results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Original image")
plt.imshow(sample_image, cmap='gray')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Max-Pooling")
plt.imshow(pooled_output[0], cmap='gray')
plt.axis('off')

plt.show()

## Concluding Remarks

In this notebook, we implemented convolution and analysed how different kernels affect the convolution operator.
We also implemented pooling operations both manually and using PyTorch's built-in modules.

Now you can start using these module, with the understanding of what they do, in the development of your own CNN.